In [4]:
# !pip install fvcore

In [3]:
#fastscnn.py
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from PIL import ImageFilter
import os
import cv2
import torchvision.transforms as tf
import shutil
import random
import gc

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Enable memory efficient mode
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False) # Changed inplace=True to inplace=False

    def forward(self, x):
        x = self.relu(self.bn1(self.depthwise(x)))
        x = self.relu(self.bn2(self.pointwise(x)))
        return x

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expand_ratio=6):
        super(InvertedResidual, self).__init__()
        self.stride = stride
        self.use_residual = stride == 1 and in_channels == out_channels

        hidden_dim = in_channels * expand_ratio
        layers = []

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=False) # Changed inplace=True to inplace=False
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=False), # Changed inplace=True to inplace=False
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        else:
            return self.conv(x)

class LearningToDownsample(nn.Module):
    def __init__(self):
        super(LearningToDownsample, self).__init__()
        self.conv = nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=False) # Changed inplace=True to inplace=False

        self.dsconv1 = DepthwiseSeparableConv(32, 48, stride=2)
        self.dsconv2 = DepthwiseSeparableConv(48, 64, stride=2)

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))  # /2
        x = self.dsconv1(x)  # /4
        x = self.dsconv2(x)  # /8
        return x

class GlobalFeatureExtractor(nn.Module):
    def __init__(self):
        super(GlobalFeatureExtractor, self).__init__()
        # Bottleneck layers for efficiency
        self.block1 = self._make_layer(64, 96, 3, stride=2)    # /16
        self.block2 = self._make_layer(96, 128, 3, stride=2)   # /32
        self.block3 = self._make_layer(128, 128, 3, stride=1)  # /32

        # Global average pooling + fc for global context
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False) # Changed inplace=True to inplace=False
        )

    def _make_layer(self, in_channels, out_channels, num_blocks, stride=1):
        layers = []
        layers.append(InvertedResidual(in_channels, out_channels, stride, expand_ratio=6))
        for _ in range(num_blocks - 1):
            layers.append(InvertedResidual(out_channels, out_channels, 1, expand_ratio=6))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.block1(x)      # /16
        x = self.block2(x)      # /32
        x = self.block3(x)      # /32

        # Global context
        global_context = self.global_pool(x)
        global_context = self.fc(global_context)
        global_context = F.interpolate(global_context, size=x.shape[2:], mode='bilinear', align_corners=False)

        return x + global_context

class FeatureFusionModule(nn.Module):
    def __init__(self, higher_in_channels, lower_in_channels, out_channels):
        super(FeatureFusionModule, self).__init__()
        self.conv_higher = nn.Sequential(
            nn.Conv2d(higher_in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.conv_lower = nn.Sequential(
            nn.Conv2d(lower_in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.conv_fusion = nn.Sequential(
            DepthwiseSeparableConv(out_channels, out_channels),
            nn.ReLU(inplace=False) # Changed inplace=True to inplace=False
        )
        self.relu = nn.ReLU(inplace=False) # Changed inplace=True to inplace=False

    def forward(self, higher_res_feature, lower_res_feature):
        lower_res_feature = F.interpolate(lower_res_feature, size=higher_res_feature.shape[2:],
                                        mode='bilinear', align_corners=False)

        higher_res_feature = self.conv_higher(higher_res_feature)
        lower_res_feature = self.conv_lower(lower_res_feature)

        fused = self.relu(higher_res_feature + lower_res_feature)
        fused = self.conv_fusion(fused)

        return fused

class Classifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(Classifier, self).__init__()
        self.dsconv1 = DepthwiseSeparableConv(in_channels, 128)
        self.dsconv2 = DepthwiseSeparableConv(128, 128)
        self.conv = nn.Conv2d(128, num_classes, 1)
        self.dropout = nn.Dropout2d(0.1)

    def forward(self, x):
        x = self.dsconv1(x)
        x = self.dsconv2(x)
        x = self.dropout(x)
        x = self.conv(x)
        return x

class FastSCNN(nn.Module):
    def __init__(self, num_classes=19):
        super(FastSCNN, self).__init__()
        self.learning_to_downsample = LearningToDownsample()
        self.global_feature_extractor = GlobalFeatureExtractor()
        self.feature_fusion = FeatureFusionModule(64, 128, 128)
        self.classifier = Classifier(128, num_classes)

    def forward(self, x):
        input_size = x.shape[2:]

        # Learning to Downsample - preserves spatial info at 1/8 resolution
        higher_res_features = self.learning_to_downsample(x)  # 64 channels, /8

        # Global Feature Extractor - extracts semantic info at 1/32 resolution
        lower_res_features = self.global_feature_extractor(higher_res_features)  # 128 channels, /32

        # Feature Fusion - combines both streams
        fused_features = self.feature_fusion(higher_res_features, lower_res_features)  # 128 channels, /8

        # Classification
        output = self.classifier(fused_features)

        # Upsample to input size
        output = F.interpolate(output, size=input_size, mode='bilinear', align_corners=False)

        return output

# Accuracy calculation (unchanged)
def cal_acc(pred_folder, gt_folder, classes=19):
    class AverageMeter(object):
        def __init__(self):
            self.reset()
        def reset(self):
            self.val, self.avg, self.sum, self.count = 0, 0, 0, 0
        def update(self, val, n=1):
            self.val = val
            self.sum += val * n
            self.count += n
            self.avg = self.sum / self.count
    def intersectionAndUnion(output, target, K, ignore_index=255):
        assert output.ndim in [1, 2, 3]
        assert output.shape == target.shape
        output = output.reshape(output.size).copy()
        target = target.reshape(target.size)
        output[np.where(target == ignore_index)[0]] = ignore_index
        intersection = output[np.where(output == target)[0]]
        area_intersection, _ = np.histogram(intersection, bins=np.arange(K + 1))
        area_output, _ = np.histogram(output, bins=np.arange(K + 1))
        area_target, _ = np.histogram(target, bins=np.arange(K + 1))
        area_union = area_output + area_target - area_intersection
        return area_intersection, area_union, area_target
    data_list = os.listdir(gt_folder)
    intersection_meter = AverageMeter()
    union_meter = AverageMeter()
    target_meter = AverageMeter()
    for image_name in data_list:
        pred = cv2.imread(os.path.join(pred_folder, image_name), cv2.IMREAD_GRAYSCALE)
        target = cv2.imread(os.path.join(gt_folder, image_name), cv2.IMREAD_GRAYSCALE)
        intersection, union, target = intersectionAndUnion(pred, target, classes)
        intersection_meter.update(intersection)
        union_meter.update(union)
        target_meter.update(target)
    iou_class = intersection_meter.sum / (union_meter.sum + 1e-10)
    mIoU = np.mean(iou_class)
    print(f'Eval result: mIoU {mIoU:.4f}.')
    return mIoU

def make_folder(dir_name):
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)

def move_folders(grey_temp, color_temp, grey_rs, color_rs):
    if os.path.exists(grey_temp):
        make_folder(grey_rs)
        for file in os.listdir(grey_temp):
            shutil.move(os.path.join(grey_temp, file), os.path.join(grey_rs, file))
        if os.path.exists(grey_temp):
            shutil.rmtree(grey_temp)
    if os.path.exists(color_temp):
        make_folder(color_rs)
        for file in os.listdir(color_temp):
            shutil.move(os.path.join(color_temp, file), os.path.join(color_rs, file))
        if os.path.exists(color_temp):
            shutil.rmtree(color_temp)

def colorize(gray, palette):
    color = Image.fromarray(gray.astype(np.uint8)).convert('P')
    color.putpalette(palette)
    return color

def get_predictions(segNet, dataFolder, device):
    gray_folder, color_folder = 'temp_grey', 'temp_color'
    listImages, gt_folder = os.listdir(os.path.join(dataFolder, "testing/image")), os.path.join(dataFolder, "testing/label")
    print('Begin testing')
    make_folder(gray_folder)
    make_folder(color_folder)
    colors_path = os.path.join(dataFolder, "colors.txt")
    colors = np.loadtxt(colors_path).astype('uint8')

    transformTest = tf.Compose([tf.ToPILImage(), tf.Resize((375, 1242)), tf.ToTensor(),
                                tf.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

    for idx in range(len(listImages)):
        img_path = os.path.join(dataFolder, "testing/image", listImages[idx])
        img = cv2.imread(img_path)[:, :, 0:3]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_height, original_width = img.shape[:2]

        if original_width <= 0 or original_height <= 0:
            print(f"Skipping image {listImages[idx]} due to invalid dimensions")
            continue

        img = transformTest(img).unsqueeze(0)

        with torch.no_grad():
            pred = segNet(img.to(device))
            pred = F.interpolate(pred, size=(original_height, original_width), mode='bilinear', align_corners=False)
            gray = pred.argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)

        color = colorize(gray, colors)
        gray_path = os.path.join(gray_folder, listImages[idx])
        color_path = os.path.join(color_folder, listImages[idx])
        cv2.imwrite(gray_path, gray)
        color.save(color_path)

        del pred
        torch.cuda.empty_cache()

    return gray_folder, color_folder

class ExpDataSet(Dataset):
    def __init__(self, dataFolder, is_train=True):
        self.is_train = is_train
        self.image_path = sorted(os.listdir(os.path.join(dataFolder, "training/image")))
        self.label_path = sorted(os.listdir(os.path.join(dataFolder, "training/label")))
        print(f'load info for {len(self.image_path)} images')
        assert len(self.image_path) == 150

        for idx in range(len(self.image_path)):
            assert self.image_path[idx] == self.label_path[idx]
            self.image_path[idx] = os.path.join(dataFolder, "training/image", self.image_path[idx])
            self.label_path[idx] = os.path.join(dataFolder, "training/label", self.label_path[idx])

        self.height = 375
        self.width = 1242

    def __getitem__(self, idx):
        img = cv2.imread(self.image_path[idx])[:, :, 0:3]
        label = cv2.imread(self.label_path[idx], 0)

        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        label = Image.fromarray(label)

        if self.is_train:
            scale = random.choice([0.75, 1.0, 1.25])
            new_h = int(self.height * scale)
            new_w = int(self.width * scale)

            img = img.resize((new_w, new_h), Image.BILINEAR)
            label = label.resize((new_w, new_h), Image.NEAREST)

            if scale > 1.0:
                i = random.randint(0, new_h - self.height)
                j = random.randint(0, new_w - self.width)
                img = img.crop((j, i, j + self.width, i + self.height))
                label = label.crop((j, i, j + self.width, i + self.height))
            else:
                img = img.resize((self.width, self.height), Image.BILINEAR)
                label = label.resize((self.width, self.height), Image.NEAREST)

            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                label = label.transpose(Image.FLIP_LEFT_RIGHT)

            color_jitter = transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)
            img = color_jitter(img)

            if random.random() < 0.3:
                img = img.filter(ImageFilter.GaussianBlur(radius=1))

            if random.random() < 0.2:
                img = transforms.functional.adjust_gamma(img, gamma=random.uniform(0.8, 1.2))

            img = img.resize((self.width, self.height), Image.BILINEAR)
            label = label.resize((self.width, self.height), Image.NEAREST)

        else:
            img = img.resize((self.width, self.height), Image.BILINEAR)
            label = label.resize((self.width, self.height), Image.NEAREST)

        img = transforms.ToTensor()(img)
        img = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))(img)
        label = torch.tensor(np.array(label), dtype=torch.long)

        return img, label

    def __len__(self):
        return len(self.image_path)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets, ignore_index=255):
        n_classes = inputs.shape[1]
        valid_mask = (targets != ignore_index)
        inputs = F.softmax(inputs, dim=1)
        safe_targets = torch.where(valid_mask, targets, torch.zeros_like(targets))
        targets_onehot = F.one_hot(safe_targets, num_classes=n_classes).permute(0, 3, 1, 2).float()
        valid_mask = valid_mask.unsqueeze(1).float()
        inputs = inputs * valid_mask
        targets_onehot = targets_onehot * valid_mask
        inputs = inputs.contiguous().view(-1, n_classes)
        targets_onehot = targets_onehot.contiguous().view(-1, n_classes)
        intersection = (inputs * targets_onehot).sum(dim=0)
        dice = (2. * intersection + self.smooth) / (inputs.sum(dim=0) + targets_onehot.sum(dim=0) + self.smooth)
        return 1 - dice.mean()

def train_model(data_dir='seg_data'):
    batch_size = 16
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    learning_rate = 2e-3
    num_epochs = 200
    start_epoch = 0
    best_mIoU = 0.0

    train_data = ExpDataSet(data_dir, is_train=True)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                            num_workers=4, pin_memory=True, drop_last=False)

    # FastSCNN model
    segNet = FastSCNN(num_classes=19).to(device)

    ckpt_path = './best_fastscnn_model.pth'
    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=device)
        segNet.load_state_dict(state, strict=True)

    # Mixed precision training
    scaler = torch.amp.GradScaler('cuda')

    # Compute class weights
    class_weights = torch.ones(19, dtype=torch.float32).to(device)
    try:
        hist = np.zeros(19, dtype=np.int64)
        for p in train_data.label_path:
            lab = cv2.imread(p, 0)
            if lab is None:
                continue
            lab = lab[lab != 255]
            if lab.size > 0:
                hist += np.bincount(lab.flatten(), minlength=19)
        freq = hist / (hist.sum() + 1e-6)
        weights = 1.0 / (np.log(1.02 + freq) + 1e-8)
        class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    except Exception:
        pass

    # Loss functions
    ce_loss = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=0.1, weight=class_weights)
    dice_loss = DiceLoss()

    def combined_loss(outputs, targets):
        return 0.8 * ce_loss(outputs, targets) + 0.2 * dice_loss(outputs, targets)

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(segNet.parameters(), lr=learning_rate, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    patience_counter = 0
    patience = 25
    mIoU_list = []

    for epoch in range(start_epoch, num_epochs):
        segNet.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for iter, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(device), labels.long().to(device)

            with torch.amp.autocast('cuda'):
                outputs = segNet(imgs)
                outputs = F.interpolate(outputs, size=labels.shape[1:], mode='bilinear', align_corners=False)
                loss = combined_loss(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(segNet.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()

            if iter % 4 == 0:
                print(f'Epoch {epoch}/{num_epochs} Iter {iter}/{len(train_loader)} Loss={loss.item():.4f}')

        scheduler.step()

        avg_train_loss = train_loss / len(train_loader)
        # print(f'Epoch {epoch} Average Training Loss: {avg_train_loss:.4f}')

        torch.cuda.empty_cache()
        gc.collect()

        # Evaluate every epoch
        if (epoch + 1) % 5 == 0:
            segNet.eval()

            with torch.no_grad():
                gray_folder, color_folder = get_predictions(segNet, data_dir, device)
                current_mIoU = cal_acc(gray_folder, os.path.join(data_dir, 'testing/label'))
                mIoU_list.append(current_mIoU)
                print(f'Epoch {epoch} mIoU={current_mIoU:.4f}')

                if current_mIoU > best_mIoU:
                    best_mIoU = current_mIoU
                    patience_counter = 0
                    torch.save(segNet.state_dict(), './best_fastscnn_model.pth')
                    move_folders(
                        gray_folder, color_folder,
                        gray_folder.replace('temp_', ''),
                        color_folder.replace('temp_', '')
                    )
                    print(f'New best model saved with mIoU: {best_mIoU:.4f}')
                else:
                    patience_counter += 1

                if os.path.exists(gray_folder):
                    shutil.rmtree(gray_folder)
                if os.path.exists(color_folder):
                    shutil.rmtree(color_folder)

        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}. Best mIoU: {best_mIoU:.4f}')
            break

        torch.cuda.empty_cache()
        gc.collect()

    print(f'Training completed. Best mIoU: {best_mIoU:.4f}')
    return segNet, best_mIoU, mIoU_list

def calculate_gflops(model, input_size=None):
    input_size = (1, 3, 375, 1242)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device).eval()
    dummy = torch.randn(input_size, device=device)

    try:
        import logging
        from fvcore.nn import FlopCountAnalysis
        logging.getLogger("fvcore.nn.jit_analysis").setLevel(logging.ERROR)
        with torch.no_grad():
            flops = FlopCountAnalysis(model, dummy).total()
        return flops / 1e9
    except Exception:
        try:
            from thop import profile
            with torch.no_grad():
                flops, _ = profile(model, inputs=(dummy,))
            return flops / 1e9
        except Exception as e:
            raise RuntimeError(
                "FLOPs computation failed: install fvcore or thop."
            ) from e

if __name__ == "__main__":
    data_dir = '/kaggle/input/seg-data/seg_data'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    weights_path = 'best_fastscnn_model.pth'

    if os.path.exists(weights_path):
        segNet = FastSCNN(num_classes=19).to(device)
        state = torch.load(weights_path, map_location=device)
        segNet.load_state_dict(state, strict=True)
        segNet.eval()

        gflops = calculate_gflops(segNet)
        print(f"gflops are {gflops}")

        with torch.no_grad():
            gray_folder, color_folder = get_predictions(segNet, data_dir, device)
            best_score = cal_acc(gray_folder, os.path.join(data_dir, 'testing/label'))

        efficiency = best_score * 100 / gflops
        print(f"Final best mIoU: {best_score:.4f}")
        print(f"Efficiency Score: {efficiency:.2f}")

        if os.path.exists(gray_folder):
            shutil.rmtree(gray_folder)
        if os.path.exists(color_folder):
            shutil.rmtree(color_folder)
    else:
        segNet, best_score, _ = train_model(data_dir)
        segNet.eval()
        gflops = calculate_gflops(segNet)
        print(f"gflops are {gflops}")
        efficiency = best_score * 100 / gflops
        print(f"Final best mIoU: {best_score:.4f}")
        print(f"Efficiency Score: {efficiency:.2f}")

load info for 150 images
Epoch 0/200 Iter 0/10 Loss=3.3702
Epoch 0/200 Iter 4/10 Loss=2.9443
Epoch 0/200 Iter 8/10 Loss=2.8213
Epoch 1/200 Iter 0/10 Loss=2.7792
Epoch 1/200 Iter 4/10 Loss=2.7402
Epoch 1/200 Iter 8/10 Loss=2.5464
Epoch 2/200 Iter 0/10 Loss=2.6547
Epoch 2/200 Iter 4/10 Loss=2.5131
Epoch 2/200 Iter 8/10 Loss=2.4501
Epoch 3/200 Iter 0/10 Loss=2.3450
Epoch 3/200 Iter 4/10 Loss=2.4350
Epoch 3/200 Iter 8/10 Loss=2.3386
Epoch 4/200 Iter 0/10 Loss=2.4537
Epoch 4/200 Iter 4/10 Loss=2.3118
Epoch 4/200 Iter 8/10 Loss=2.3528
Begin testing
Eval result: mIoU 0.1816.
Epoch 4 mIoU=0.1816
New best model saved with mIoU: 0.1816
Epoch 5/200 Iter 0/10 Loss=2.3607
Epoch 5/200 Iter 4/10 Loss=2.2640
Epoch 5/200 Iter 8/10 Loss=2.3277
Epoch 6/200 Iter 0/10 Loss=2.2218
Epoch 6/200 Iter 4/10 Loss=2.2785
Epoch 6/200 Iter 8/10 Loss=2.2414
Epoch 7/200 Iter 0/10 Loss=2.2748
Epoch 7/200 Iter 4/10 Loss=2.2251
Epoch 7/200 Iter 8/10 Loss=2.2312
Epoch 8/200 Iter 0/10 Loss=2.2378
Epoch 8/200 Iter 4/10 Loss